# 1h. Animating a Geometry Optimization

This notebook turns a <span style="font-variant: small-caps">Psi4</span> geometry
optimization into a multi-frame `.xyz` **trajectory** that Avogadro (or `py3Dmol`)
will animate.

The short version: **don't route this through `cclib` or Open Babel.**  Both are
third-party parsers chasing a moving target (Psi4's print format changes between
releases, and the file the Python API writes is not the same file the `psi4`
executable writes).  Psi4 already hands you the geometries directly, so two
dependency-free routes are better:

| | Route | Use when |
|---|---|---|
| **A** | `psi4.optimize(..., return_history=True)` | you are running the optimization now |
| **B** | parse the geometry blocks out of the output file | you only have an old `output.dat` |

Route **A** is the recommended one.  Route **B** is ~40 lines of plain Python with
no dependencies, and it works on any output file you already have sitting around.

A multi-frame `.xyz` is just single-frame `.xyz` blocks concatenated — atom count,
comment line, then one line per atom — repeated once per step.  Avogadro detects the
repeats and gives you the animation slider automatically; no special format needed.

## What actually goes wrong

There are **three independent bugs** in the obvious cclib version of this script,
and they stack — fixing one just exposes the next.  All three were reproduced
against cclib 1.8.1.

**1. `ccread()` returns `None`, so the "missing metadata" patch is a red herring.**
`cclib.io.ccread()` does not know the file is Psi4 output; it *sniffs* the opening
lines for the Psi4 banner to pick a parser.  That banner is printed by the `psi4`
**executable** — a file written from the Python API with
`psi4.core.set_output_file(...)` starts straight in at `=> Loading Basis Set <=`
with no banner at all.  Sniffing fails and you get:

```
Unable to determine the type of logfile ['water_optimization.out'], try the fallback mechanism
Could not import `openbabel`, fallback mechanism might not work.
```

then `None`.  The next line, `data.metadata = {}`, raises
`AttributeError: 'NoneType' object has no attribute 'metadata'` — which looks like
a metadata problem but is really "no parser was selected."  When the parser *is*
named explicitly, `data.metadata` is present and populated, so that patch is
never needed.

Note the second line of that message: **cclib's own fallback calls Open Babel.**
If you were seeing Open Babel tracebacks as well as cclib ones, they are very
likely the same root cause surfacing through cclib's fallback path, not two
separate problems.

**2. `outputfilename=` is not a parameter, so no file is ever written.**  The
signature is

```python
ccwrite(ccobj, outputtype=None, outputdest=None, indices=None, terse=False, returnstr=False, *args, **kwargs)
```

The keyword is **`outputdest`**.  `outputfilename="trajectory.xyz"` is swallowed by
`**kwargs` and silently ignored; with `outputdest` left at `None`, `ccwrite`
*returns* the XYZ as a string instead of writing it.  Nothing hits disk — and
`print("Successfully exported trajectory.xyz...")` still runs, which is what makes
this one so annoying to spot.

**3. `allgeom` defaults to `False`, so you get one frame, not a trajectory.**
cclib's XYZ writer writes only the *last* geometry unless told otherwise.  Even
with bugs 1 and 2 fixed, `trajectory.xyz` would be a single static structure with
nothing to animate.  You need `allgeom=True`.

**Also: `psi4.core.clean()` does not flush the output file.**  It removes scratch
files — a different job entirely.  The call that flushes and closes the output
handler is `psi4.core.close_outfile()` (confirm what your build exposes with
`[m for m in dir(psi4.core) if "outfile" in m]`).  Reading the output file before
it is flushed gives you a truncated parse, which can look like yet another parser
bug.

The corrected cclib version is at the bottom of this notebook and it does work.
But every one of these is a symptom of round-tripping data through a text file and
a third-party parser when you already hold the geometries in memory — which is
what the two routes below do instead.

## Route A — `return_history=True` (recommended)

Two details do all the work here, and they are the reason the history looked
awkward to use:

1. **The coordinates are in bohr, not ångström.**  Multiply by
   `psi4.constants.bohr2angstroms`.  Skip this and the animation looks right but
   every bond is ~1.9× too long.
2. **The element symbols are not in the history** — the history is geometry only.
   Get them from the molecule with `mol.symbol(i)`.

The type of each history entry (`psi4.core.Matrix`, `ndarray`, or a nested list)
varies with the Psi4 version, which is what you ran into.  `np.asarray(...)` after
an optional `.to_array()` handles all three, so the helper below does not care which
one your build produces.

One more thing that is easy to miss: **freeze the frame before optimizing.**

In [ ]:
import numpy as np
import psi4

psi4.set_memory("2 GB")
psi4.set_output_file("h2o_opt.dat", False)

h2o = psi4.geometry("""
0 1
O
H 1 0.96
H 1 0.96 2 104.5
""")

# IMPORTANT: without these two lines Psi4 re-centers and re-orients the molecule
# at every step, so the animation shows the molecule tumbling around in space
# instead of the bonds relaxing. This is the single most common reason an
# otherwise-correct trajectory "looks wrong" in Avogadro.
h2o.fix_com(True)
h2o.fix_orientation(True)

psi4.set_options({"basis": "cc-pvdz", "g_convergence": "gau_tight"})

energy, wfn, history = psi4.optimize(
    "scf", molecule=h2o, return_wfn=True, return_history=True
)

print("converged energy:", energy)
print("history keys:    ", list(history.keys()))
print("steps:           ", len(history["coordinates"]))

Now the conversion. This is the "bit more hacking" — it is about fifteen lines.

In [ ]:
def history_to_xyz(molecule, history, path):
    """Write a Psi4 optimization history out as a multi-frame .xyz trajectory.

    Parameters
    ----------
    molecule : psi4.core.Molecule
        The molecule that was optimized (supplies the element symbols).
    history : dict
        The third return value of psi4.optimize(..., return_history=True).
    path : str
        Destination .xyz file.
    """
    bohr2ang = psi4.constants.bohr2angstroms
    symbols = [molecule.symbol(i).capitalize() for i in range(molecule.natom())]
    energies = history.get("energy", [])

    with open(path, "w") as fh:
        for n, geom in enumerate(history["coordinates"]):
            # psi4.core.Matrix, ndarray, or nested list -- all three land here
            raw = geom.to_array() if hasattr(geom, "to_array") else geom
            coords = np.asarray(raw, dtype=float).reshape(len(symbols), 3) * bohr2ang

            e = f"  E = {energies[n]:.10f} Eh" if n < len(energies) else ""
            fh.write(f"{len(symbols)}\nstep {n}{e}\n")
            for s, (x, y, z) in zip(symbols, coords):
                fh.write(f"{s:<4s}{x:>18.10f}{y:>18.10f}{z:>18.10f}\n")
    return path


history_to_xyz(h2o, history, "h2o_opt_traj.xyz")
print(open("h2o_opt_traj.xyz").read()[:400])

That file opens in Avogadro with the animation controls enabled
(**File → Open**, then the play/slider in the Animation tool).

### Aside: letting Psi4 write the trajectory for you

Psi4's optimizer can emit the `.xyz` trajectory itself, which skips the helper
entirely:

```python
psi4.set_options({"print_trajectory_xyz_file": True})
```

This is an `optking` option, so whether it is available — and where it drops the
file — depends on your Psi4/optking version.  Run the cell below to find out; if it
raises, just use `history_to_xyz` above, which works on every version.

In [ ]:
try:
    psi4.set_options({"print_trajectory_xyz_file": True})
    print("available -- Psi4 will write a trajectory .xyz during the next optimize()")
    print("look for it near:", psi4.core.get_writer_file_prefix(h2o.name()) + ".xyz")
except Exception as exc:
    print("not available in this build:", exc)
    print("-> use history_to_xyz() instead")
finally:
    psi4.set_options({"print_trajectory_xyz_file": False})

## Route B — parse the output file directly

Use this when you already have an `output.dat` from a previous run and don't want
to repeat the optimization.  It reads the `Geometry (in Angstrom), charge = ...`
blocks that Psi4 prints at the top of every step — a format that has been stable
for many releases — and it needs nothing but the standard library.

It also handles the three things that trip up the generic parsers:

* blocks printed **in bohr** (the "Final optimized geometry" reprint often is),
* the **duplicate** frames from Psi4 reprinting the same geometry, and
* pulling the **energy** that belongs with each geometry.

If you want the raw, undeduplicated list of every printed geometry, pass
`dedupe=False`.

In [ ]:
import re

BOHR2ANG = 0.52917720859

_GEOM_RE = re.compile(
    r"Geometry \(in (Angstrom|Bohr)\), charge\s*=\s*-?\d+, multiplicity\s*=\s*\d+:"
)
_ATOM_RE = re.compile(
    r"^\s*([A-Za-z][A-Za-z]?[0-9]*(?:\([A-Za-z]{1,2}\))?)\s+"
    r"(-?\d+\.\d+)\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)"
)
_ENER_RE = re.compile(
    r"(?:@\S*\s*Final Energy:|Total Energy\s*=|Current energy\s*:)\s*(-?\d+\.\d+)"
)


def parse_psi4_opt(path, dedupe=True, tol=1e-8):
    """Pull every geometry (and the energy that follows it) out of a Psi4 output file.

    Returns a list of dicts:
        {"symbols": [...], "coords": [[x, y, z], ...] in Angstrom, "energy": float | None}
    """
    with open(path) as fh:
        lines = fh.readlines()

    frames, i = [], 0
    while i < len(lines):
        m = _GEOM_RE.search(lines[i])
        if not m:
            # between geometry blocks: look for the energy of the frame we just read
            if frames and frames[-1]["energy"] is None:
                e = _ENER_RE.search(lines[i])
                if e:
                    frames[-1]["energy"] = float(e.group(1))
            i += 1
            continue

        # skip the "Center X Y Z Mass" header and its dashed rule
        unit, j = m.group(1), i + 1
        while j < len(lines) and "---------" not in lines[j] and not _GEOM_RE.search(lines[j]):
            j += 1
        j += 1

        syms, xyz = [], []
        while j < len(lines):
            a = _ATOM_RE.match(lines[j])
            if not a:
                break
            syms.append(a.group(1).capitalize())
            xyz.append([float(a.group(k)) for k in (2, 3, 4)])
            j += 1

        if syms:
            if unit == "Bohr":
                xyz = [[c * BOHR2ANG for c in row] for row in xyz]
            frames.append({"symbols": syms, "coords": xyz, "energy": None})
        i = max(j, i + 1)

    return _dedupe(frames, tol) if dedupe else frames


def _dedupe(frames, tol):
    """Psi4 reprints the same geometry several times; collapse consecutive repeats."""
    out = []
    for f in frames:
        if out:
            p = out[-1]
            same = (p["symbols"] == f["symbols"] and
                    all(abs(a - b) < tol
                        for r1, r2 in zip(p["coords"], f["coords"])
                        for a, b in zip(r1, r2)))
            if same:
                if p["energy"] is None:      # keep the energy off the duplicate
                    p["energy"] = f["energy"]
                continue
        out.append(f)
    return out


def frames_to_xyz(frames, path):
    """Write parsed frames as a multi-frame .xyz trajectory."""
    with open(path, "w") as fh:
        for n, f in enumerate(frames):
            e = "" if f["energy"] is None else f"  E = {f['energy']:.10f} Eh"
            fh.write(f"{len(f['symbols'])}\nstep {n}{e}\n")
            for s, (x, y, z) in zip(f["symbols"], f["coords"]):
                fh.write(f"{s:<4s}{x:>18.10f}{y:>18.10f}{z:>18.10f}\n")
    return path

In [ ]:
frames = parse_psi4_opt("h2o_opt.dat")
print(f"{len(frames)} unique geometries")

for n, f in enumerate(frames):
    e = "n/a" if f["energy"] is None else f"{f['energy']:.8f}"
    print(f"  step {n:2d}   E = {e}")

frames_to_xyz(frames, "h2o_opt_traj_parsed.xyz")

A quick sanity check worth running before you trust any trajectory:
the energy should go **down** monotonically, and the last frame should match the
optimized geometry Psi4 reported.

In [ ]:
energies = [f["energy"] for f in frames if f["energy"] is not None]

if len(energies) > 1:
    deltas = np.diff(energies)
    print("energy lowered at every step:", bool(np.all(deltas <= 1e-10)))
    print(f"total relaxation: {energies[-1] - energies[0]:.6f} Eh "
          f"({(energies[-1] - energies[0]) * 627.509:.2f} kcal/mol)")

last = np.array(frames[-1]["coords"])
print("\nfinal O-H bond lengths (Angstrom):")
for i in range(1, len(last)):
    print(f"  O-{frames[-1]['symbols'][i]}{i}: {np.linalg.norm(last[i] - last[0]):.6f}")

## Checking the animation without leaving the notebook

Before exporting to Avogadro it is convenient to confirm the trajectory looks
sensible here.  `py3Dmol` animates a multi-frame `.xyz` string directly — note
`addModelsAsFrames` (plural, "as frames"), not `addModel`, which would show only
the first geometry.

In [ ]:
import py3Dmol

traj = open("h2o_opt_traj.xyz").read()

view = py3Dmol.view(width=500, height=400)
view.addModelsAsFrames(traj, "xyz")
view.setStyle({"stick": {"radius": 0.12}, "sphere": {"scale": 0.25}})
view.animate({"loop": "forward", "interval": 400, "reps": 0})
view.zoomTo()
view.show()

## Appendix — the corrected `cclib` version

For completeness, here is the cclib route with all three bugs fixed.  It works;
it is just more moving parts than Routes A and B, and it will keep needing
attention as Psi4's print format and cclib's parser drift apart.

The three changes:

1. `Psi4(fn).parse()` instead of `ccread(fn)` — name the parser, don't let it guess.
2. `outputdest=` instead of `outputfilename=`.
3. `allgeom=True` — without it you get one frame.

The `data.metadata` patch is dropped; it is not needed once the parser is named.

In [ ]:
import psi4
from cclib.parser import Psi4          # name the parser explicitly
from cclib.io import ccwrite

output_filename = "water_optimization.out"
psi4.core.set_output_file(output_filename, False)

mol = psi4.geometry("""
0 1
O  0.0  0.0  0.0
H  0.0  0.8  0.6
H  0.0 -0.8  0.6
""")
mol.fix_com(True)            # otherwise the molecule tumbles through the animation
mol.fix_orientation(True)

energy, history = psi4.optimize("scf/cc-pvdz", molecule=mol, return_history=True)

# Flush the output file. psi4.core.clean() does NOT do this -- it clears scratch.
psi4.core.close_outfile()

data = Psi4(output_filename).parse()
print("geometries parsed:", data.atomcoords.shape)      # (nsteps, natom, 3)

ccwrite(
    data,
    outputtype="xyz",
    outputdest="trajectory.xyz",   # NOT outputfilename= -- that is silently ignored
    allgeom=True,                  # NOT optional -- without it you get only the last frame
)

with open("trajectory.xyz") as fh:
    frames = fh.read().count("[Geometry")
print(f"wrote trajectory.xyz with {frames} frames")

## Summary

* Multi-frame `.xyz` is just concatenated `.xyz` blocks; Avogadro animates it with
  no extra work.
* **Route A** (`return_history=True`) is the one to reach for: convert bohr →
  ångström, take the symbols from the molecule, and `np.asarray(...)` whatever the
  history hands you.  You already have the history — the file round-trip buys you
  nothing.
* **Route B** parses an existing output file with no dependencies at all.
* Call `fix_com(True)` and `fix_orientation(True)` before optimizing, or the
  molecule will tumble through the animation.
* If you do use cclib: name the parser, use `outputdest=`, and pass `allgeom=True`.
* `psi4.core.clean()` clears scratch; `psi4.core.close_outfile()` flushes output.